# LLM-based classifier

Install requirements

In [ ]:
# !pip3 install -q transformers torch accelerate sentencepiece

Initialization

In [ ]:
import os
import re
import torch
import pandas as pd
from typing import List, Dict, Tuple
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

In [ ]:


# # =========================================================
# # Configuration
# # =========================================================
# MODEL_NAME = "google/flan-t5-small"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MAX_INPUT_LENGTH = 512
# MAX_NEW_TOKENS = 10


# # =========================================================
# # Load Data
# # =========================================================
# train_df = pd.read_csv('data/6/train.csv')

# # =========================================================
# # Prompt Templates
# # =========================================================
# PROMPT_TEMPLATES = {
#     "prompt_1": """
#     You are a text classification system.

#     Classify the sentiment of the following text into exactly one label from:
#     {labels}

#     Return only one label.

#     Text:
#     {text}

#     Label:
#     """.strip(),

#     "prompt_2": """
#     Classify the sentiment of this text as one of these labels: {labels}.

#     Text: {text}

#     Answer with only one label.
#     """.strip(),

#         "prompt_3": """
#     Read the text and predict its sentiment.

#     Possible labels: {labels}

#     Text:
#     {text}

#     Output only the correct label.
#     """.strip(),

#     "prompt_4": """
#     You must perform sentiment classification.

#     Allowed labels:
#     {labels}

#     Rules:
#     - Choose one label only
#     - Do not explain

#     Text:
#     {text}

#     Sentiment:
#     """.strip(),

#     "prompt_5": """
#     Determine whether the sentiment of the following text is:
#     {labels}

#     Text:
#     {text}

#     Return only the label.
#     """.strip(),
#     }



/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Functions

In [ ]:
def normalize_label(raw_output: str, valid_labels: List[str]) -> str:
    """
    Convert raw model output into one valid label.

    Strategy:
    1. Strip spaces and lowercase the output
    2. Return exact match if available
    3. Otherwise, search for the first valid label inside the output
    4. Return 'UNKNOWN' if no valid label is found
    """
    cleaned_output = str(raw_output).strip().lower()
    normalized_labels = [str(label).strip().lower() for label in valid_labels]

    if cleaned_output in normalized_labels:
        return cleaned_output

    for label in normalized_labels:
        if re.search(rf"\b{re.escape(label)}\b", cleaned_output):
            return label

    return "UNKNOWN"


def load_hf_model_and_tokenizer(
    model_name: str,
    device: str
):
    """
    Load the appropriate Hugging Face tokenizer and model.

    Uses:
    - AutoModelForSeq2SeqLM for T5 / FLAN-T5 style models
    - AutoModelForCausalLM for GPT-style models

    Returns:
        tokenizer, model, model_type
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lowered_name = model_name.lower()
    is_seq2seq = any(name_part in lowered_name for name_part in ["t5", "flan"])

    model_dtype = torch.float16 if device == "cuda" else torch.float32

    if is_seq2seq:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=model_dtype
        )
        model_type = "seq2seq"
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=model_dtype
        )
        model_type = "causal"

    model.to(device)
    model.eval()

    return tokenizer, model, model_type


def build_prompt(
    prompt_template: str,
    text: str,
    valid_labels: List[str]
) -> str:
    """
    Format one prompt using the provided template.

    Expected placeholders in prompt_template:
    - {text}
    - {labels}
    """
    labels_text = ", ".join([str(label) for label in valid_labels])
    return prompt_template.format(text=str(text), labels=labels_text)


def predict_single_text(
    text: str,
    prompt_template: str,
    valid_labels: List[str],
    tokenizer,
    model,
    model_type: str,
    max_input_length: int,
    max_new_tokens: int,
    device: str
) -> Tuple[str, str]:
    """
    Predict one label for one text using one prompt template.

    Returns:
        raw_output: raw generated text from the model
        predicted_label: normalized predicted label
    """
    prompt = build_prompt(prompt_template, text, valid_labels)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    if model_type == "seq2seq":
        raw_output = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()
    else:
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

    predicted_label = normalize_label(raw_output, valid_labels)
    return raw_output, predicted_label


def evaluate_prompt_predictions(
    y_true: List[str],
    y_pred: List[str]
) -> Dict[str, float]:
    """
    Calculate evaluation metrics for one prompt.
    """
    y_true_clean = [str(label).strip().lower() for label in y_true]
    y_pred_clean = [str(label).strip().lower() for label in y_pred]

    return {
        "accuracy": accuracy_score(y_true_clean, y_pred_clean),
        "f1_macro": f1_score(y_true_clean, y_pred_clean, average="macro", zero_division=0)
    }

def run_llm_prompt_evaluation(
    model_name: str,
    prompt_templates: Dict[str, str],   # <- DICT now
    max_input_length: int,
    max_new_tokens: int,
    input_df: pd.DataFrame,
    target_column_name: str,
    text_column_name: str,
    valid_labels: List[str],
    device: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run LLM-based classification for all rows and all prompt templates.

    Parameters:
        model_name:
            Hugging Face model name.
        prompt_templates:
            Dictionary of prompt templates. Each template must use:
            - {text}
            - {labels}
        max_input_length:
            Maximum number of input tokens.
        max_new_tokens:
            Maximum number of generated tokens.
        input_df:
            Input dataframe containing text and true labels.
        target_column_name:
            Column name containing true labels.
        text_column_name:
            Column name containing input text.
        valid_labels:
            List of valid class labels.
        device:
            'cpu' or 'cuda'.

    Returns:
        modified_input_df:
            Copy of input dataframe with added prediction columns:
            out_label_Prompt_1, out_label_Prompt_2, ...
        results_df:
            Dataframe with one row per prompt and evaluation metrics.
    """

    modified_input_df = input_df.copy()

    tokenizer, model, model_type = load_hf_model_and_tokenizer(
        model_name=model_name,
        device=device
    )

    y_true = modified_input_df[target_column_name].astype(str).str.lower().tolist()
    texts = modified_input_df[text_column_name].astype(str).tolist()

    results = []

    # Loop over dictionary
    for idx, (prompt_name, prompt_template) in enumerate(prompt_templates.items(), start=1):

        # Required column format (assignment requirement)
        column_name = f"out_label_Prompt_{idx}"

        predictions = []

        for text in texts:
            _, pred_label = predict_single_text(
                text=text,
                prompt_template=prompt_template,
                valid_labels=valid_labels,
                tokenizer=tokenizer,
                model=model,
                model_type=model_type,
                max_input_length=max_input_length,
                max_new_tokens=max_new_tokens,
                device=device
            )
            predictions.append(pred_label)

        # Add predictions to dataframe
        modified_input_df[column_name] = predictions

        # Evaluate
        scores = evaluate_prompt_predictions(y_true=y_true, y_pred=predictions)

        results.append({
            "model_name": model_name,
            "prompt_name": prompt_name,   # <- keep dictionary key
            "prediction_column": column_name,
            "accuracy": scores["accuracy"],
            "f1_macro": scores["f1_macro"]
        })

    results_df = pd.DataFrame(results)

    return modified_input_df, results_df

def save_results(predicted_df: pd.DataFrame, 
                 results_df: pd.DataFrame, 
                 prompt_templates: Dict[str, str], 
                 base_dir: str = 'data/outputs') -> None:
    """
    Save the predicted dataframe, results dataframe, and prompt templates 
    to a new timestamped folder.

    Args:
        predicted_df (pd.DataFrame): DataFrame containing predictions.
        results_df (pd.DataFrame): DataFrame containing evaluation metrics.
        prompt_templates (Dict[str, str]): Dictionary of prompt templates.
        base_dir (str): Base directory where results will be saved.
    """
    # Create a new folder with an incremented number
    existing_folders = [int(folder) for folder in os.listdir(base_dir) if folder.isdigit()]
    next_folder = str(max(existing_folders) + 1) if existing_folders else '1'
    new_folder_path = os.path.join(base_dir, next_folder)
    os.makedirs(new_folder_path)

    # Save the predicted dataframe
    predicted_df.to_csv(os.path.join(new_folder_path, 'predicted_df.csv'), index=False)

    # Save the results dataframe
    results_df.to_csv(os.path.join(new_folder_path, 'results_df.csv'), index=False)

    # Save the prompt templates as a CSV file
    prompt_templates_df = pd.DataFrame(list(prompt_templates.items()), columns=['Prompt Name', 'Prompt Template'])
    prompt_templates_df.to_csv(os.path.join(new_folder_path, 'prompt_templates.csv'), index=False)

Run for a prompt and LLM

In [ ]:
# Load Data
train_df = pd.read_csv('data/6/train.csv')
df = train_df.head(100)

# Configuration
MODEL_NAME = "google/flan-t5-small"
PROMPT_TEMPLATES = {
    "prompt_1": """
    You are a text classification system.

    Classify the sentiment of the following text into exactly one label from:
    {labels}

    Return only one label.

    Text:
    {text}

    Label:
    """.strip(),

    "prompt_2": """
    Classify the sentiment of this text as one of these labels: {labels}.

    Text: {text}

    Answer with only one label.
    """.strip(),

        "prompt_3": """
    Read the text and predict its sentiment.

    Possible labels: {labels}

    Text:
    {text}

    Output only the correct label.
    """.strip(),

    "prompt_4": """
    You must perform sentiment classification.

    Allowed labels:
    {labels}

    Rules:
    - Choose one label only
    - Do not explain

    Text:
    {text}

    Sentiment:
    """.strip(),

    "prompt_5": """
    Determine whether the sentiment of the following text is:
    {labels}

    Text:
    {text}

    Return only the label.
    """.strip(),
    }

In [ ]:
modified_df, results_df = run_llm_prompt_evaluation(
    model_name=MODEL_NAME,
    prompt_templates=PROMPT_TEMPLATES,
    max_input_length=512,
    max_new_tokens=10,
    input_df=df,
    target_column_name="sentiment",
    text_column_name="text",
    valid_labels=["positive", "negative"],
    device="cpu"
)

results_df

,model_name,prompt_name,prediction_column,accuracy,f1_macro
0,google/flan-t5-small,prompt_1,out_label_Prompt_1,0.89,0.872552
1,google/flan-t5-small,prompt_2,out_label_Prompt_2,0.87,0.851852
2,google/flan-t5-small,prompt_3,out_label_Prompt_3,0.84,0.824176
3,google/flan-t5-small,prompt_4,out_label_Prompt_4,0.90,0.883123
4,google/flan-t5-small,prompt_5,out_label_Prompt_5,0.70,0.692118


In [ ]:
# save_results(predicted_df=modified_df, results_df=results_df, prompt_templates=PROMPT_TEMPLATES)